# Research source: Historical parent library and case audit

**Route:** archival methodology, not the default frozen-deployment quick start.  
**Original uploaded notebook:** `01_LP_Run_Guide_and_Audit(1).ipynb`.  
**Purpose:** Validates the eleven-unit case and reuses the historical parent library. No new synthetic library should be substituted.

## Before executing
The public optimizer uses the supplied numerical includes and frozen coefficients. It **does not need this notebook to be rerun**. Start with `notebooks/00_Quick_Start.ipynb` and `GAMS_LaPalma/START_LAPALMA.gms` instead.

Full execution of this research notebook needs: Historical input workbook; LP_Case.json; original source hashes. These archival dependencies are not all in the lightweight public package. The exact original case/protocol bytes have not been fabricated. Original source-hash checks are intentionally strict and may reject comment-sanitized public copies; archival replay uses the exact private originals and the author's matching data archive.

## Governing model and units
UC dispatch and reserve are in MW; surviving inertia is in MW s; gain is in MW/Hz. The primary frequency criterion is a 46.5-Hz minimum at 50 Hz. The historical notebook branch remains 47 Hz. The SODE model uses one frequency state and two states per surviving governor; its response is limited above by individual headroom. The feature vector order is `loss_mw, inertia_mws, gain_mw_hz, reserve_mw`. A calibrated score becomes an affine GAMS inequality after inverse standardization and a probability-logit shift.

## Algorithm, interpretation and verification
Read the original Markdown and the stage annotations below in order. The detailed derivations and symbol-to-source mapping are in `docs/Mathematical_Formulation/Technical_Manual.pdf`. A grouping or calibration diagnostic is not a physical risk guarantee. Do not transfer a previous day's state through rounded Excel output; use native GDX.

**Curation status:** notebook structure, imports, Python syntax and exact preservation of all original code-cell text are checked. Full production execution is **NOT RUN**. Source-only functions may be exercised by the small verification notebooks using explicitly labelled fixtures; that does not validate the unavailable full experiment.

Original outputs and execution counts were removed from this public copy to avoid stale results and private workstation paths. The separately supplied private original backup preserves the original bytes.


# 01 — La Palma: source audit and run guide

**Work in this separate `LaPalma_SEGAN` project, through Anaconda/Jupyter.** No terminal, GAMS Python API, custom Python module, Bernstein transcription or spatial-network calculation is required. GAMS is run manually in its IDE. Preserve the original inputs and caches.

The governing sequence is **01 → 02 → 03 → GAMS IDE `Run_LaPalma_Seasons.gms` → 04**. Four representative seasonal weeks are independent: winter, spring, summer and autumn. Each consists of seven consecutive 24-hour solves with same-method state transfer. Money is EUR; native source coefficients are kEUR and are converted explicitly.

**Analytical comparator now included:** Raja's supplied `DM_SP.gms` frequency block is installed as a separate SP include, using the study's **3.0 Hz drop / 47.0 Hz boundary** instead of the source's 3.5 Hz. Its breakpoints and selector equations are retained. The common chronological UC, ramps and costs are the corrected formulation, not a replay of the entire historical program. The source SP damping credit and the declared SODE damping convention differ; both are preserved and explicitly reported. Read `RUN_LAPALMA.md` before interpreting model-to-model differences.


## What this first step establishes
The source contains 20,000 rows, but repeated dispatches are not independent experiments. A **parent** is the whole eleven-unit pre-event dispatch. Its online-unit outages stay in the same data partition. No chronological constraints are claimed for these synthetic snapshots. Generator 11 is absent from the source library by the historical economic-domain choice; we do not infer that it is physically unavailable.

## Shared mathematical definitions
All functions are inside this notebook. Their source text is identical across the four notebooks so dataset labels and post-UC assessments use the same model.

Let $\Delta f=f-50$ (negative after a loss). The response law is
$$\frac{2E}{f_0}\dot{\Delta f}=\sum_i \min(r_i,\rho_i)-L-D_{Hz}\Delta f.$$
For each surviving generator,
$$\frac{r_i(s)}{\Delta f(s)}=-G_i\frac{1+b_i s}{1+a_{1i}s+a_{2i}s^2},\qquad G_i=K_iM_i/f_0.$$
We use a stable state-space realization; the internal response is capped only at delivered output, not by modifying the ODE solver state. No deadband, anti-windup or field-identification evidence is invented.

**Damping convention:** $D_{Hz}=P_D(0.01)/50$ MW/Hz, retaining Raja's source convention. This is numerically different from the frozen IEEE campaign's $0.01P_D$. Consistent methodology does not mean silently assigning identical numerical assumptions to both systems. Parent `load` equals thermal production in the supplied source; that proxy is recorded, whereas seasonal schedules use their actual gross demand.


### Source code stage 4

**Algorithm objects:** `find_project`, `sha_file`, `hash_object`, `atomic_json`, `atomic_csv`, `write_excel`, `excel_values`, `number`, `read_matrix`, `read_pairs`, `assert_sources`, `prepare_parents`, `event_inputs`, `event_key`, `integrate_event`, `evaluate_event`, `cache_open`, `cache_save`, `simulate_parent_library`.

These definitions implement the original routines; inspect their input validation and return fields before calling them. Defining a function does not itself verify its numerical behavior.

**Production workload:** this stage may fit models or process a large event/campaign library. It was not run for packaging. Do not execute it merely to use the released frozen coefficients.

**Verification:** original code-cell text is unchanged and syntax-checked. Printed values and figures become new evidence only after their required inputs have been supplied and execution actually succeeds. Preserve caches/receipts and distinguish a failed calculation from an unsafe physical label.

In [ ]:
# Shared definitions are embedded verbatim in the notebooks; no custom .py is needed.
from pathlib import Path
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED
from xml.etree import ElementTree as ET
import csv, hashlib, json, math, os, re, shutil, sys, time, tempfile
import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp
from scipy.optimize import minimize_scalar
from scipy.special import expit


def find_project():
    here = Path.cwd().resolve()
    for p in (here, here.parent):
        if (p/'Data_LaPalma/LP_Case.json').is_file() and (p/'GAMS_LaPalma').is_dir():
            return p
    raise FileNotFoundError('Open this notebook in LaPalma_SEGAN/Notebooks_LaPalma, not the IEEE project.')

PROJECT = find_project()
NB = PROJECT/'Notebooks_LaPalma'
DATA = PROJECT/'Data_LaPalma'
GAMS = PROJECT/'GAMS_LaPalma'
EVID = PROJECT/'Evidence'; EVID.mkdir(exist_ok=True)
CASE = json.loads((DATA/'LP_Case.json').read_text(encoding='utf-8'))
IDS = CASE['ids']; NGEN = len(IDS)
if IDS != [f'i{j}' for j in range(1, 12)]:
    raise ValueError('Expected the original ordered eleven-unit La Palma fleet.')
PMAX = np.array(CASE['pmax']); PMIN = np.array(CASE['pmin'])
HS = np.array(CASE['inertia_s']); MBASE = np.array(CASE['mbase'])
KG = np.array(CASE['gain_pu']); A1 = np.array(CASE['a1'])
A2 = np.array(CASE['a2']); B1 = np.array(CASE['b1'])
F0 = float(CASE['f0_hz']); FMIN = float(CASE['nadir_min_hz'])
if not (np.isclose(F0,50.) and np.isclose(F0-FMIN,3.)):
    raise ValueError('This SEGAN La Palma design requires nominal50 Hz and a 3-Hz drop (47-Hz threshold).')
EUNIT = HS*MBASE
GAIN = KG*MBASE/F0  # MW/Hz; source gain K*Mbase = F0*GAIN.
for vector in (PMAX, PMIN, HS, MBASE, KG, A1, A2, B1):
    if vector.shape != (11,) or not np.isfinite(vector).all():
        raise ValueError('Every unit-parameter vector must contain exactly eleven finite entries.')
CORE_FEATURES = ['loss_mw', 'inertia_mws', 'gain_mw_hz', 'reserve_mw']
NUMERIC = dict(method='LSODA', rtol=1e-8, atol=1e-10, max_step=.20,
               output_step=.02, duration=60., extension=120.,
               stop_hz=40., settle_range_hz=.005, settle_slope_hz_s=1e-4,
               nadir_refine_tol_hz=1e-3, boundary_hz=1e-4)
DYNAMIC_VERSION = 'LP_UNIT_ID_UPPER_CAP_HZ_SOURCE_DAMP_v3'
MODEL_SPEC = {k:CASE[k] for k in ['ids','pmax','inertia_s','mbase','gain_pu','a1','a2','b1',
                                      'f0_hz','damping_fraction_per_hz','nadir_min_hz',
                                      'rocof_max_hz_s','qss_drop_hz']}
MODEL_SPEC.update(version=DYNAMIC_VERSION, limiter='minimum(internal_response, unit_headroom)',
                  numeric=NUMERIC, units='Hz / MW / MW*s / MW-per-Hz')

def sha_file(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda:handle.read(1024*1024),b''): h.update(block)
    return h.hexdigest()

def hash_object(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,separators=(',',':'),allow_nan=False).encode()).hexdigest()

MODEL_ID = hash_object(MODEL_SPEC)[:16]
WORK = DATA/'Corrected'/MODEL_ID; WORK.mkdir(parents=True,exist_ok=True)

def atomic_json(path, value):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp')
    tmp.write_text(json.dumps(value,indent=2,sort_keys=True,default=lambda x:x.item() if isinstance(x,np.generic) else str(x)),encoding='utf-8')
    os.replace(tmp,path)

def atomic_csv(frame,path):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp');frame.to_csv(tmp,index=False);os.replace(tmp,path)

def write_excel(path, sheets):
    """CSV/checkpoints are authoritative. Write/validate a NEW workbook, then replace.
    A Windows-locked old workbook is not deleted and never costs another simulation.
    All sheet names are keyword arguments (pandas 3-compatible).
    """
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.stem+'_writing_'+str(time.time_ns())+'.xlsx')
    try:
        with pd.ExcelWriter(tmp,engine='openpyxl') as writer:
            for name,frame in sheets.items():
                if not isinstance(frame,pd.DataFrame):frame=pd.DataFrame(frame)
                frame.to_excel(writer,sheet_name=name[:31],index=False)
                ws=writer.sheets[name[:31]];ws.freeze_panes='A2'
                ws.auto_filter.ref=ws.dimensions
                for c in ws[1]: c.font=c.font.copy(bold=True)
                for column_index in range(1,ws.max_column+1):
                    col=[ws.cell(row=row_index,column=column_index) for row_index in range(1,min(100,ws.max_row)+1)]
                    samples=[len(str(c.value)) for c in col if c.value is not None]
                    ws.column_dimensions[col[0].column_letter].width=min(38,max(12,max(samples,default=12)+2))
        with ZipFile(tmp) as z:
            if z.testzip() is not None or 'xl/workbook.xml' not in z.namelist():raise IOError('Invalid Excel export')
        try: os.replace(tmp,path)
        except PermissionError:
            path=path.with_name(path.stem+'_RECOVERED_'+str(time.time_ns())+'.xlsx');os.replace(tmp,path)
            print('The prior workbook is locked; saved a fresh workbook:',path)
    except Exception:
        print('Excel export failed. Saved CSVs/caches are retained; rerun only the output cell.')
        raise
    return path

# Reader uses stored cell values, including GAMS EPS, with no Excel automation.
def excel_values(path, sheet):
    ns={'x':'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}
    rid='{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id'
    with ZipFile(path) as z:
        strings=[]
        if 'xl/sharedStrings.xml' in z.namelist():
            strings=[''.join(x.itertext()) for x in ET.fromstring(z.read('xl/sharedStrings.xml')).findall('x:si',ns)]
        rel={x.attrib['Id']:x.attrib['Target'] for x in ET.fromstring(z.read('xl/_rels/workbook.xml.rels'))}
        sheets=ET.fromstring(z.read('xl/workbook.xml')).find('x:sheets',ns)
        target=next((rel[s.attrib[rid]] for s in sheets if s.attrib['name']==sheet),None)
        if target is None:raise KeyError(f'{Path(path).name}: missing sheet {sheet}')
        target=target.lstrip('/') if target.startswith('/') else 'xl/'+target
        rows=[]
        for row in ET.fromstring(z.read(target)).findall('x:sheetData/x:row',ns):
            d={}
            for c in row:
                col=0
                for a in re.match(r'[A-Z]+',c.attrib['r']).group():col=col*26+ord(a)-64
                v=c.find('x:v',ns);typ=c.attrib.get('t')
                if typ=='s':value=strings[int(v.text)] if v is not None else None
                elif typ=='inlineStr':value=''.join(c.find('x:is',ns).itertext())
                elif v is None:value=None
                elif typ in ('str','e'):value=v.text
                else:
                    try:value=float(v.text)
                    except (ValueError,TypeError):value=v.text
                d[col-1]=value
            rows.append(d)
        width=max((max(r,default=-1)+1 for r in rows),default=0)
        return [[r.get(j) for j in range(width)] for r in rows]

def number(value):
    if value is None or value=='':return 0.
    if isinstance(value,str) and value.strip().lower() in ('eps','epsilon'):return 0.
    out=float(value)
    if not math.isfinite(out):raise ValueError(f'Nonfinite native number: {value}')
    return out

def read_matrix(path,sheet,row_ids,col_ids):
    data=excel_values(path,sheet);result=np.zeros((len(row_ids),len(col_ids)))
    headers=next((r for r in data if sum(str(x) in col_ids for x in r)>=1),None)
    if headers is None:raise ValueError(f'{sheet}: no expected column IDs')
    positions={str(x):j for j,x in enumerate(headers) if str(x) in col_ids}
    if set(positions)!=set(col_ids):raise ValueError(f'{sheet}: incomplete column identifiers')
    mapped={str(r[0]):r for r in data if r and str(r[0]) in row_ids}
    if set(mapped)!=set(row_ids):raise ValueError(f'{sheet}: incomplete row identifiers')
    for i,key in enumerate(row_ids):
        for j,col in enumerate(col_ids):result[i,j]=number(mapped[key][positions[col]])
    return result

def read_pairs(path,sheet,expected=None):
    result={}
    for row in excel_values(path,sheet):
        if len(row)>=2 and row[0] is not None and (expected is None or str(row[0]) in expected):
            try:result[str(row[0])]=number(row[1])
            except (ValueError,TypeError):continue
    return result

def assert_sources():
    for relative,expected in CASE['source_hashes'].items():
        actual=sha_file(PROJECT/relative)
        if actual!=expected:raise ValueError(f'Original source changed: {relative}; do not reuse this case identity.')


def prepare_parents(seed=20260920):
    """Reuse the historical pool. No new A-LHS generation or hidden i11 restriction."""
    path=DATA/'A-LHS_newDataretry.xlsx'
    power=np.asarray(excel_values(path,'generation'),float)
    demand=np.asarray(excel_values(path,'load'),float).reshape(-1)
    if power.shape[1]!=11 or len(demand)!=len(power):
        raise ValueError('The source already has eleven columns; never append a twelfth dummy unit.')
    if np.any(demand<16-1e-6) or np.any(demand>36+1e-6):raise ValueError('Historical parent-load domain differs from 16--36 MW')
    online=power>1e-8
    violations=(power<-1e-7).any(1)|(power>PMAX+1e-7).any(1)|((power<PMIN-1e-7)&online).any(1)
    violations|=np.abs(power.sum(1)-demand)>1e-6
    if violations.any():raise ValueError(f'{violations.sum()} source parents fail declared static screens; review, do not silently repair.')
    # A group is the complete pre-event dispatch (all its outages remain together).
    gid=[hashlib.sha256(np.round(p,6).astype('<f8').tobytes()).hexdigest()[:24] for p in power]
    frame=pd.DataFrame(power,columns=IDS);frame.insert(0,'parent_id',gid)
    frame.insert(0,'source_row',np.arange(1,len(power)+1));frame['demand_mw']=demand
    frame['original_filter']='historical A-LHS cost-screened source; selection bias retained'
    frame['static_screen_pass']=True
    # Equal dispatches belong to one group even if a future source has another demand.
    groups=sorted(set(gid));rng=np.random.default_rng(seed);order=np.array(groups,dtype=object)[rng.permutation(len(groups))]
    ends=np.round(np.cumsum([.60,.15,.10,.15])*len(groups)).astype(int)
    assignment={};start=0
    for role,end in zip(['development','calibration','selection','test'],ends):
        assignment.update({str(g):role for g in order[start:end]});start=end
    frame['role']=frame.parent_id.map(assignment)
    # Duplicate physically identical rows are not counted as independent examples.
    unique=frame.drop_duplicates(['parent_id','demand_mw']).copy().reset_index(drop=True)
    unique['demand_band']=np.minimum(39, np.floor((unique.demand_mw-16)/.5+1e-7).astype(int))
    blocks=np.asarray(CASE['blocks']);slopes=np.asarray(CASE['slopes_keur_per_mwh'])
    rem=unique[IDS].to_numpy().copy();cost=(rem>1e-8)@np.asarray(CASE['no_load_keur_h'])
    for j in range(3):q=np.minimum(rem,blocks[:,j]);cost+=(q*slopes[:,j]).sum(1);rem-=q
    unique['screen_cost_keur_h']=cost
    atomic_csv(frame,WORK/'Parent_source_map.csv');atomic_csv(unique,WORK/'Parents.csv')
    receipt=dict(source_rows=len(frame),unique_physical_parents=len(unique),unique_groups=len(groups),
        duplicate_rows=len(frame)-len(unique),candidate_outage_rows_source=int(online.sum()),
        unique_parent_outages=int((unique[IDS].to_numpy()>1e-8).sum()),unit11_online=int(online[:,10].sum()),
        role_counts=unique.groupby('role').size().to_dict(),source_sha256=sha_file(path),model_id=MODEL_ID,
        scope='Static generation/load/headroom/inertia inputs; no chronology, no LP network screening',
        demand_definition='Historical load equals total thermal output: treated as supplied reference damping load. Seasonal deployment uses actual gross demand. Not evidence of identical distributions.')
    atomic_json(WORK/'Parent_receipt.json',receipt)
    return unique,receipt


def event_inputs(power, demand_mw, outage_index, commitment=None):
    power=np.asarray(power,float).copy();demand_mw=float(demand_mw);j=int(outage_index)
    if power.shape!=(11,) or not np.isfinite(power).all() or demand_mw<0:raise ValueError('Invalid event input')
    on=(power>1e-8) if commitment is None else np.rint(np.asarray(commitment,float)).astype(bool)
    if on.shape!=(11,) or not on[j] or power[j]<=1e-8:raise ValueError('Outage must remove an online producing unit')
    if np.any(power<-1e-6) or np.any(power>PMAX*on+1e-6):raise ValueError('Invalid pre-event generator outputs')
    survive=on.copy();survive[j]=False
    idx=np.flatnonzero(survive);E=float(EUNIT[survive].sum());rho=np.maximum(PMAX-power,0)[idx]
    return dict(power=power,online=on,survive=survive,index=idx,E=E,rho=rho,
        gain=GAIN[idx],a1=A1[idx],a2=A2[idx],b1=B1[idx],loss=float(power[j]),
        damping=demand_mw*CASE['damping_fraction_per_hz'],demand=demand_mw,outage=j)


def event_key(power,demand_mw,outage_index,commitment=None):
    on=(np.asarray(power)>1e-8) if commitment is None else np.rint(commitment).astype(bool)
    identity=dict(model_id=MODEL_ID,power=np.round(power,9).tolist(),demand=round(float(demand_mw),9),
                  outage=IDS[int(outage_index)],commitment=on.astype(int).tolist())
    return hash_object(identity)


def integrate_event(power,demand_mw,outage_index,commitment=None,refined=False,return_trace=False):
    v=event_inputs(power,demand_mw,outage_index,commitment)
    if v['E']<=0:raise ValueError('No surviving inertia; no fictitious epsilon inertia is inserted')
    n=len(v['index']);rtol=NUMERIC['rtol']*(.1 if refined else 1.)
    atol=NUMERIC['atol']*(.1 if refined else 1.)
    step=NUMERIC['output_step']*(.5 if refined else 1.)
    mx=NUMERIC['max_step']*(.5 if refined else 1.)
    def response(y):return np.minimum(v['gain']*(y[1:1+n]+v['b1']*y[1+n:]),v['rho'])
    def rhs(t,y):
        df=y[0];q=y[1:1+n];dq=y[1+n:]
        dfdt=F0/(2*v['E'])*(response(y).sum()-v['loss']-v['damping']*df)
        ddq=(-q-v['a1']*dq-df)/v['a2']
        return np.concatenate(([dfdt],dq,ddq))
    def stop(t,y):return F0+y[0]-NUMERIC['stop_hz']
    stop.terminal=True;stop.direction=-1
    y0=np.zeros(1+2*n);duration=NUMERIC['duration'];pieces=[];start=0.;state=y0
    while True:
        sol=solve_ivp(rhs,(start,duration),state,method=NUMERIC['method'],rtol=rtol,atol=atol,
                      max_step=mx,dense_output=True,events=stop)
        if not sol.success or not np.isfinite(sol.y).all():raise RuntimeError(sol.message)
        end=float(sol.t[-1]);grid=np.linspace(start,end,max(2,int(np.ceil((end-start)/step))+1))
        Y=sol.sol(grid);frequency=F0+Y[0];pieces.append((grid,Y,sol))
        censored=bool(sol.t_events[0].size)
        tail=grid>=max(start,end-5.)
        slope=float(rhs(end,Y[:,-1])[0]);range_=float(np.ptp(frequency[tail]))
        settled=(not censored and end>=10 and range_<=NUMERIC['settle_range_hz'] and abs(slope)<=NUMERIC['settle_slope_hz_s'])
        if censored or settled or duration>=NUMERIC['extension']:break
        start=end;state=Y[:,-1];duration=NUMERIC['extension']
    times=np.concatenate([g if k==0 else g[1:] for k,(g,_,__) in enumerate(pieces)])
    states=np.concatenate([y if k==0 else y[:,1:] for k,(_,y,__) in enumerate(pieces)],axis=1)
    freq=F0+states[0];nadir=float(freq.min());tn=float(times[freq.argmin()])
    # Refine each sampled local minimum using the dense interpolation of its own interval.
    for grid,Y,sol in pieces:
        f=F0+Y[0]
        for k in np.flatnonzero((f[1:-1]<f[:-2])&(f[1:-1]<=f[2:]))+1:
            opt=minimize_scalar(lambda t:float(F0+sol.sol(t)[0]),bounds=(grid[k-1],grid[k+1]),method='bounded',options={'xatol':1e-9})
            if opt.fun<nadir:nadir=float(opt.fun);tn=float(opt.x)
    rtotal=np.array([response(y).sum() for y in states.T])
    record=dict(minimum_observed_hz=nadir,time_min_s=tn,endpoint_hz=float(freq[-1]),
        initial_rocof_hz_s=float(rhs(0,y0)[0]),settled=bool(settled),censored=bool(censored),
        simulated_seconds=float(times[-1]),tail_range_hz=range_,tail_slope_hz_s=slope,
        loss_mw=v['loss'],inertia_mws=v['E'],gain_mw_hz=float(v['gain'].sum()),
        reserve_mw=float(v['rho'].sum()),gain_source=float(F0*v['gain'].sum()),demand_mw=float(demand_mw))
    return record,dict(time_s=times,frequency_hz=freq,response_mw=rtotal) if return_trace else None


def evaluate_event(power,demand_mw,outage_index,commitment=None):
    key=event_key(power,demand_mw,outage_index,commitment);started=time.perf_counter()
    base=dict(event_key=key,model_id=MODEL_ID,outage_unit=IDS[int(outage_index)],outage_index=int(outage_index))
    try:
        coarse,_=integrate_event(power,demand_mw,outage_index,commitment)
        fine,_=integrate_event(power,demand_mw,outage_index,commitment,refined=True)
        delta=abs(coarse['minimum_observed_hz']-fine['minimum_observed_hz'])
        agree=(coarse['minimum_observed_hz']<FMIN)==(fine['minimum_observed_hz']<FMIN)
        numerical=bool(agree and delta<=NUMERIC['nadir_refine_tol_hz'] and coarse['censored']==fine['censored'])
        witnessed=bool(fine['minimum_observed_hz']<FMIN-NUMERIC['boundary_hz'])
        resolved=bool(numerical and fine['settled'] and not fine['censored'])
        if numerical and witnessed:status='UNSAFE_OBSERVED';label=-1
        elif resolved and fine['minimum_observed_hz']>=FMIN+NUMERIC['boundary_hz']:status='SAFE_CONDITIONAL';label=1
        else:status='REVIEW';label=None
        base.update(fine,status=status,y_safe=label,numerical_refinement_ok=numerical,nadir_refinement_abs_hz=delta,
            nadir_crossing_observed=witnessed,resolved_nadir=resolved,
            rocof_crossing=bool(abs(fine['initial_rocof_hz_s'])>CASE['rocof_max_hz_s']+1e-6),
            qss_unresolved=not fine['settled'],qss_crossing=bool(fine['settled'] and abs(F0-fine['endpoint_hz'])>CASE['qss_drop_hz']+1e-6),
            review_reason='' if status!='REVIEW' else ('numerical_refinement' if not numerical else 'unsettled_or_boundary'),
            error='')
    except Exception as exc:
        base.update(status='FAILED_RETAINED',y_safe=None,numerical_refinement_ok=False,nadir_crossing_observed=False,
                    resolved_nadir=False,qss_unresolved=True,error=repr(exc))
    base['calculation_seconds']=time.perf_counter()-started
    return base


def cache_open(path):
    path=Path(path);cache={}
    if path.exists():
        lines=path.read_text(encoding='utf-8').splitlines()
        for i,line in enumerate(lines):
            if not line.strip():continue
            try:row=json.loads(line)
            except json.JSONDecodeError:
                if i==len(lines)-1:
                    backup=path.with_suffix('.truncated_backup_'+str(time.time_ns()))
                    shutil.copyfile(path,backup)
                    path.write_text('\n'.join(lines[:i])+'\n',encoding='utf-8');break
                raise ValueError('Corrupt cache before its last line; preserve file for inspection')
            if row.get('model_id')!=MODEL_ID:raise ValueError('Cache physics/numerics identity differs; never reuse these labels')
            cache[row['event_key']]=row
    return cache

def cache_save(handle,row):
    handle.write(json.dumps(row,allow_nan=False,default=lambda x:x.item() if isinstance(x,np.generic) else str(x))+'\n');handle.flush()


def simulate_parent_library(parents, limit=None, n_jobs=2):
    """One cached physical calculation per unique dispatch/load/outage/model identity.
    Worker processes (not threads) isolate LSODA's integrator state. Results are
    appended before Excel/plotting. Interrupting a batch only loses unfinished tasks.
    """
    from joblib import Parallel, delayed
    chosen=parents if limit is None else parents.iloc[:int(limit)]
    cachepath=WORK/'Label_cache.jsonl';cached=cache_open(cachepath);rows=[];pending={};descriptors=[]
    for record in chosen.to_dict('records'):
        power=np.array([record[i] for i in IDS]);demand_mw=float(record['demand_mw'])
        for j in np.flatnonzero(power>1e-8):
            key=event_key(power,demand_mw,j)
            descriptors.append((key,record))
            if key not in cached:pending[key]=(power,demand_mw,int(j))
    jobs=list(pending.values());started=time.perf_counter();added=0
    print(len(descriptors),'event records;',len(jobs),'uncached integrations; worker processes:',n_jobs,flush=True)
    with cachepath.open('a',encoding='utf-8') as handle:
        if jobs:
            with Parallel(n_jobs=max(1,int(n_jobs)),backend='loky',batch_size=1) as pool:
                for start in range(0,len(jobs),32):
                    done=pool(delayed(evaluate_event)(p,d,j) for p,d,j in jobs[start:start+32])
                    for r in done:cache_save(handle,r);cached[r['event_key']]=r;added+=1
                    if added%128==0 or added==len(jobs):
                        print(f'{added}/{len(jobs)} newly calculated; {time.perf_counter()-started:.1f}s',flush=True)
    for key,record in descriptors:
        rows.append({**cached[key],'parent_id':record['parent_id'],'source_row':record['source_row'],
                     'role':record['role'],'demand_band':record['demand_band'],'screen_cost_keur_h':record['screen_cost_keur_h']})
    frame=pd.DataFrame(rows);atomic_csv(frame,WORK/'LP_SODE_Labels.csv')
    receipt=dict(model_id=MODEL_ID,parents=len(chosen),events=len(frame),new_events=added,
        status_counts=frame.status.value_counts().to_dict(),labels_sha256=sha_file(WORK/'LP_SODE_Labels.csv'),
        parent_sha256=sha_file(WORK/'Parents.csv'),production_complete=(len(chosen)==len(parents)),
        source_scope='fixed historical parent pool; all online single-generator events, no spatial extension')
    atomic_json(WORK/'Labels_receipt.json',receipt)
    return frame,receipt


## Check the four seasons and original sources
This is data preparation only; no simulations or optimization are launched.

### Source code stage 6

This cell uses objects created in preceding cells. Check the immediately preceding source explanation and the declared input files before execution.

**Verification:** original code-cell text is unchanged and syntax-checked. Printed values and figures become new evidence only after their required inputs have been supplied and execution actually succeeds. Preserve caches/receipts and distinguish a failed calculation from an unsafe physical label.

In [ ]:
assert_sources()
parents, parent_receipt = prepare_parents()
season_rows=[]
for season in CASE['seasons']:
    demand=np.asarray(CASE['demand'][season]);res=np.asarray(CASE['res'][season])
    assert demand.shape==(7,24) and res.shape==(7,24)
    assert np.all(demand>0) and np.all(res>=0)
    season_rows.append(dict(season=season,hours=demand.size,mean_demand_mw=float(demand.mean()),peak_demand_mw=float(demand.max()),mean_available_res_mw=float(res.mean())))
season_table=pd.DataFrame(season_rows)
print('Original source rows:',parent_receipt['source_rows'])
print('Unique parents:',parent_receipt['unique_physical_parents'])
print('Unique all-online outage records:',parent_receipt['unique_parent_outages'])
print('Generator i11 source commitments:',parent_receipt['unit11_online'])
print('All eleven generators remain available in GAMS. Out-of-training-support schedules will be flagged.')
print('Original .inc files unchanged. No new A-LHS library generated.')
display(season_table)
atomic_csv(season_table,EVID/'LP_Seasonal_Inputs.csv')
atomic_json(EVID/'LP_Source_Audit.json',parent_receipt)


## Optional readable audit workbook
CSV and JSON above already exist. This cell can be rerun independently if Excel is open/locked.

### Source code stage 8

This cell uses objects created in preceding cells. Check the immediately preceding source explanation and the declared input files before execution.

**Verification:** original code-cell text is unchanged and syntax-checked. Printed values and figures become new evidence only after their required inputs have been supplied and execution actually succeeds. Preserve caches/receipts and distinguish a failed calculation from an unsafe physical label.

In [ ]:
write_excel(EVID/'LP_Input_Audit.xlsx',{'Season_profiles':season_table,'Source_counts':pd.DataFrame([parent_receipt]).astype(str)})

## Next
Open Notebook 02. Keep this folder arrangement. The source design uses a 47.0 Hz threshold (3.0 Hz drop), 2.5 Hz/s initial RoCoF criterion and 0.5 Hz QSS deviation. These are declared study criteria; the dataset itself does not establish relay settings or field validation.